# Pendulum

### **Introduction**



### **What is it?**



### **Example**



# Simulation

## Setup

Installing packages (for Google Colab). If this notebook is opened in Google Colab then some packages must be installed to run the code!\
Then importing the scene and plot settings.

In [ ]:
#@title Run to install MuJoCo and `dm_control` for Google Colab

IS_COLAB = 'google.colab' in str(get_ipython())
if IS_COLAB:
    # download the repository
    !git clone https://github.com/commanderxa/alphalabs.git
    from alphalabs.mechanics.setup import install_packages_colab
    install_packages_colab()
    import alphalabs.mechanics.plot
    from alphalabs.mechanics.scene import Scene
else:
    import os, sys
    module_path = os.path.abspath(os.path.join(".."))
    if module_path not in sys.path:
        sys.path.append(module_path)
    # import the scene
    from scene import Scene
    import plot

## Import

Import all required packages to preform simulations. Packages include simulation engine, plotting libraries and other ones necessary for computations.

In [ ]:
%env MUJOCO_GL=egl

# simulation
from dm_control import mjcf

# for video recording
import mediapy

# computations
import numpy as np

# plot charts
import seaborn as sns
import matplotlib.pyplot as plt

## Initial Conditions

In this block constants are defined. They impact the environment, rendering and objects directly.

In [ ]:
# global
viscosity = 0.00002  # air resistance
density = 1.2  # air density

# simulation constants
# lengths = [i * 0.01 for i in range(40, 60, 2)]
lengths = [0.25]
angle: float = 10  # initial angle of release (in degrees)
# mass: float = 0.5  # kg
mass: float = 1  # kg
gap: float = 0.05  # gap between pendulums (in meters)

is_ideal = False

# rendering
width = 1280
height = 720
dpi = 600
duration = 1 * 10  # (seconds)
framerate = 60  # (Hz)

## Model

### Objects of Interest

This class defines the object of our interest, a `pendulum`. Here we write what is this object (box), what can it do (swing).

In [ ]:
class SpringMassPendulum(object):
    """Spring Mass Pendulum MJCF model for vertical oscillations"""

    def __init__(
        self,
        mass: float = 1.0,
        spring_constant: float = 100.0,
        natural_length: float = 0.25,
        rgba: list = [1, 0, 0, 1],
    ) -> None:

        self.model = mjcf.RootElement()
        self.mass = mass
        self.spring_constant = spring_constant
        self.natural_length = natural_length

        self.pendulum = self.model.worldbody.add("body", name="pendulum", pos=[0, 0, 0], euler=[0, 0, 0])
        self.anchor = self.pendulum.add("site", name="anchor", pos=[0, 0, 0], size=[0.01])

        self.bob = self.pendulum.add(
            "body", name="bob", pos=[0, 0, -self.natural_length], euler=[0, 0, 0]
        )
        self.joint = self.bob.add("joint", name="mass_joint", type="slide", axis=[0, 0, 1])
        self.sphere = self.bob.add(
            "geom", name="sphere", size=[0.025], pos=[0, 0, 0], rgba=rgba, mass=mass
        )
        self.hook = self.bob.add("site", name="hook", pos=[0, 0, 0], size=[0.001])

        self.tendon = self.model.tendon.add("spatial", stiffness=self.spring_constant, springlength=f"{self.natural_length}")
        if not is_ideal:
            self.tendon.damping = 0.5
        self.tendon.width = 0.0025
        self.tendon.add("site", site="anchor")
        self.tendon.add("site", site="hook")

### World Model

Collecting everything into one general model

In [ ]:
class Model(object):

    def __init__(self) -> None:
        self.model = mjcf.RootElement(model="model")

        # set render info
        self.model.visual.__getattr__("global").offheight = height
        self.model.visual.__getattr__("global").offwidth = width

        # set the simulation constants
        if is_ideal:
            self.model.option.viscosity = 0
            self.model.option.density = 0
        else:
            self.model.option.viscosity = viscosity
            self.model.option.density = density

        # create the scene (ground)
        self.scene = Scene(length=10)
        self.scene_site = self.model.worldbody.add("site", pos=[0, 0, -0.5])
        self.scene_site.attach(self.scene.model)

        # add pendulum
        pendulum = SpringMassPendulum(mass)
        pendulum_site = self.model.worldbody.add("site", pos=[0, 0, 0])
        pendulum_site.attach(pendulum.model)

        # define camera position
        cam_x = 0
        cam_y = 3.0 * max(lengths)  # Pull back farther if needed
        # cam_z = 3 * max(lengths)
        cam_z = 3.25 * 0.25
        cam_z = -0.3
        # define camera
        self.camera = self.model.worldbody.add(
            "camera",
            name="front",
            pos=[cam_x, cam_y, cam_z],
            euler=[-90, 0, 180],
            mode="fixed",
        )

## Simulation

Initializing the `physics` of the simulation

In [ ]:
model = Model().model
physics = mjcf.Physics.from_mjcf_model(model)

First of all, the environment must be verified by rendering a picture

In [ ]:
mediapy.show_image(physics.render(height=320, width=640, camera_id=0))
mediapy.write_image("../../output/pendulum_spring_preview.png", physics.render(width=width, height=height, camera_id=0))

### Simulation Loop

In [ ]:
frames = []
timevals = []
velocity = []
position = []

while physics.time() < duration:
    timevals.append(physics.time())
    velocity.append(physics.named.data.qvel.copy())
    position.append(physics.named.data.geom_xpos.copy())

    if len(frames) < physics.time() * framerate:
        pixels = physics.render(width=width, height=height, camera_id=0)
        frames.append(pixels)

    physics.step()

In [ ]:
mediapy.show_video(frames, fps=framerate)

In [ ]:
video_name = f"pendulum_spring" 
if not IS_COLAB: video_name = f"../../output/" + video_name
if is_ideal: video_name += "_ideal" 
mediapy.write_video(video_name + ".mp4", images=frames, fps=framerate)

## Simulation Data Visualization

Convert data into numpy array to have more features

In [ ]:
timevals = np.array(timevals)
velocity = np.array(velocity).squeeze(-1)
position = np.array(position)

In [ ]:
position = position[:, -1][:, -1]

In [ ]:
cm = 1 / 2.54
figsize = (8.5 * cm, 3.75 * cm)
fig, ax = plt.subplots(ncols=2, figsize=figsize, dpi=dpi, sharey=True)
fig.subplots_adjust(wspace=0.2 * cm)

ax[0].plot(velocity, timevals, linewidth=0.5, color="tab:blue")
ax[0].set_xlabel('Velocity, [m/s]', fontsize=7, labelpad=2)
ax[0].set_ylabel('Time, [s]', fontsize=7, labelpad=2)
ax[0].invert_yaxis()
ax[0].tick_params(labelsize=6, which="both", direction="in", top=True, right=True, length=2)

ax[1].plot(position, timevals, linewidth=0.5, color="tab:blue")
ax[1].set_xlabel('Position, [m]', fontsize=7, labelpad=2)
# ax[1].set_ylabel('Time, [s]', fontsize=7, labelpad=2)
# ax[1].invert_yaxis()
ax[1].tick_params(labelsize=6, which="both", direction="in", top=True, right=True, length=2)

chart_name = "pendulum_spring"
if not IS_COLAB: chart_name = "../../output/" + chart_name
if is_ideal: chart_name += "_ideal"
plt.savefig(f"{chart_name}.png", bbox_inches='tight')

In [ ]:
# dir = f"../../errors/pendulum/"
# position_name = f"position_spring"
# if is_ideal: 
#     position_name += "_ideal"
# position_save = np.concat([position.reshape(-1, 1), timevals.reshape(-1, 1)], axis=1)
# np.save(f"{dir+position_name}", position_save)